# Emulating a Year 1 Visit Image Sequence

For the Rubin Science Platform at <a href="https://data.lsst.cloud">data.lsst.cloud</a><br>
Data Release: <a href="https://dp1.lsst.io/">Data Preview 1 (DP1)</a> <br>

**Learning Objective:** Select a set of DP1 visit images that can emulate an expected Year 1 sequence, with correct filter coverage and reasonable image quality distribution.

**LSST Data Products:** `visit_image`, `CcdVisit`

**Packages:** `lsst.daf.butler`, 

**Credit:** This notebook was created by Phil Marshall (SLAC National Accelerator Laboratory) based on the Rubin Observatory DP1 tutorial [202_2_Visit_images.ipynb](https://github.com/lsst/tutorial-notebooks/blob/main/DP1/200_Data_Products/202_Images/202_2_Visit_images.ipynb)

## 1. Introduction

We want to simulate the appearance a set of plausible lensed AGN in the LSST Year 1 images. We can use the DP1 visit images for this. In this notebook we select the right number of visit images, across all 6 bands, with the best image quality available (as expected during Y1). We need these visit images to overlap closely on the sky, to make source injection efficient. For this selection, we should not need to get the images themselves - we can use the `CcdVisit` table to investigate field overlap and PSF FWHM (image quality). 

Tutorial notes:
* A `visit_image` is a fully processed, calibrated, science-ready astronomical image from a single detector (a single CCD), obtained at a given time in a single band.
* Metadata for visits (observations) and `visit_images` are available in the `Visit` and `CcdVisit` TAP tables, respectively.

### 1.1. Import packages

From the LSST Science Pipelines import the packages for the Butler, the TAP service, 2-dimensional sky geometry, and for image display.
Also import standard `astropy` and `numpy` packages.

In [ ]:
from lsst.daf.butler import Butler, Timespan
from lsst.rsp import get_tap_service
import lsst.geom
import lsst.afw.display as afwDisplay
from astropy.time import Time
import numpy as np
import matplotlib.pyplot as plt
from lsst.utils.plotting import (get_multiband_plot_colors,
                                 get_multiband_plot_symbols)

### 1.2. Define parameters and functions

Instantiate the Butler.

In [ ]:
butler = Butler("dp1", collections="LSSTComCam/DP1")
assert butler is not None

Create an instance of the TAP service, and assert that it exists.

In [ ]:
service = get_tap_service("tap")
assert service is not None

We'll be making some multi-band plots later:

In [ ]:
filter_names = ['u', 'g', 'r', 'i', 'z', 'y']
filter_colors = get_multiband_plot_colors()
filter_symbols = get_multiband_plot_symbols()

## 2. Querying the `CcdVisit` Table

Tutorial notes:
> Image metadata for individual `visit_images` is stored in the `CcdVisit` table, which is accessed via TAP. The `CcdVisit` table includes measured image properties such as the size of the PSF,
the zeropoint, and the sky background.

We'll start by exploring the ECDFS field, which has the most visits and hence the highest chance of yielding a lot of high image quality exposures. 

Table 1 of the DP1 paper, https://rtn-095.lsst.io/, gives the field center for the ECDFS.

In [ ]:
ra0 = 53.160
dec0 = -28.100

### 2.1 Selecting the Visits

Now we query the `CcdVisit` table for single sensor images that contain the field center coordinates. 

Since we want the source injection and reprocessing to be efficient, we want sensors that overlap quite closely. Each 4k by 4k sensor is 4000*0.2/3600 = 0.22 deg across. If we require that the centroid of each visit image be within 25% of a sensor width, 0.05 deg, from the field center, that should give us a sky patch that is 0.1 deg across to inject lenses into. Such a 360 arcsec square injection region can support a 30x30 grid of 900 lenses, each separated by 12 arcsec.

We'll also include a constraint that the measured seeing be less than 1.3 arcseconds.

In [ ]:
# query = "SELECT visitId, detector, expMidptMJD, band, seeing, ra, dec, skyRotation, llcra,llcdec, ulcra,ulcdec, urcra,urcdec, lrcra,lrcdec " \
#         "FROM dp1.CcdVisit "\
#         "WHERE CONTAINS(POINT('ICRS', 53.16, -28.10), " \
#         "POLYGON('ICRS', llcra,llcdec, ulcra,ulcdec, urcra,urcdec, lrcra,lrcdec)) = 1 " \
#         "AND seeing < 1.3"

query = "SELECT visitId, detector, expMidptMJD, band, seeing, ra, dec, skyRotation, llcra,llcdec, ulcra,ulcdec, urcra,urcdec, lrcra,lrcdec " \
        "FROM dp1.CcdVisit "\
        "WHERE CONTAINS(POINT('ICRS', ra, dec), " \
        "CIRCLE('ICRS', 53.16, -28.10, 0.05)) = 1 " \
        "AND seeing < 1.3"

job = service.submit_job(query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()

In [ ]:
assert job.phase == 'COMPLETED'
visits = job.fetch_result().to_table()

Order the results by `expMidptMJD` (the MJD at the midpoint of the exposure) and display them.

In [ ]:
visits.sort('expMidptMJD')

In [ ]:
visits

In [ ]:
job.delete
del query

Let's count visits by band, and then plot histograms of image quality.

In [ ]:
x = dict()
for filt in filter_names:
    fx = np.where(visits['band'] == filt)[0]
    x[filt] = len(fx)
x

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 3))

for filt in filter_names:
    fx = np.where(visits['band'] == filt)[0]
    if len(fx) > 0:
        ax.plot(visits['expMidptMJD'][fx], visits['seeing'][fx],
                filter_symbols[filt], ms=5, mew=0, alpha=0.4,
                color=filter_colors[filt], label=filt)

ax.set_xlabel('MJD')
ax.set_ylabel('Image Quality (PDF FWHM) / arcsec')
ax.legend(loc='upper right', handletextpad=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5))

data = {}
for filt in filter_names:
    fx = np.where(visits['band'] == filt)[0]
    if len(fx) > 0:
        data[filt] = visits['seeing'][fx]
    else:
        data[filt] = 0.0

plt.hist([data['u'],data['g'],data['r'],data['i'],data['z'],data['y']], bins=10, 
            stacked=True, color=[filter_colors['u'],filter_colors['g'],filter_colors['r'],filter_colors['i'],filter_colors['z'],filter_colors['y']], 
            label=['u','g','r','i','z','y'])

ax.set_xlabel('Image Quality (PDF FWHM) / arcsec')
ax.legend(loc='upper right', handletextpad=0)
plt.tight_layout()
plt.show()

The main problem here is the lack of u-band visits. Where are they?